# Train a Tabular Classifier — LightGBM (simple)

**Maintained by:** IGNODE  
**Last verified:** 2026-05-30 against LightGBM 4.5, scikit-learn 1.5  
**Runtime:** under 1 minute on Colab's free CPU tier

Train a tabular classification model using **LightGBM** — fast gradient-boosted trees that work well out of the box for most tabular data. This notebook is **linear** (no branching, no AutoML) so you can read it end-to-end. Perfect as a learning reference or a starting point for your own variant.

If you want to compare algorithms or run AutoML, use `train_tabular_classifier.ipynb` instead.

## What you'll need

- A `.csv` file with a header row, one column per feature, and one label column
- A Google account

## Output

`model.onnx` + `class_labels.json` + `feature_columns.json` — drop into IGNODE → ML Factory → Custom Models → + Upload ML Model.

## 1. Install pinned dependencies

In [ ]:
!pip install --quiet \
    lightgbm==4.5.0 \
    scikit-learn==1.5.2 \
    onnx==1.21.0 \
    onnxmltools==1.16.0 \
    onnxconverter-common==1.14.0

import lightgbm as lgb
print(f'LightGBM: {lgb.__version__}')

## 2. Upload your CSV

Drag your CSV into the file panel on the left, or use the picker below.

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
csv_path = next(iter(uploaded.keys()))
df = pd.read_csv(csv_path)
print(f'Loaded {csv_path}: {df.shape[0]} rows x {df.shape[1]} columns')
display(df.head())

## 3. Settings

Edit these to match your dataset.

In [ ]:
# ───────── EDIT THESE ─────────
LABEL_COLUMN = 'target'   # e.g. 'Species', 'Anomaly', 'Churned'

# LightGBM hyperparameters (defaults work well for most datasets)
N_ESTIMATORS = 200        # number of boosting rounds
LEARNING_RATE = 0.05      # smaller = slower but often more accurate
MAX_DEPTH = -1            # -1 = no limit (LightGBM default)

TEST_SIZE = 0.2           # fraction of data held out for evaluation
RANDOM_SEED = 42          # locks the result for reproducibility
# ──────────────────────────────

if LABEL_COLUMN not in df.columns:
    raise ValueError(f"Label column '{LABEL_COLUMN}' not in CSV. Available: {list(df.columns)}")

## 4. Prep the data

Auto-rename column names with spaces / punctuation (IGNODE rejects those at upload). Encode string class labels as integers. Split into train + test sets.

In [ ]:
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def normalize(name):
    return re.sub(r'[^A-Za-z0-9_-]+', '_', name).strip('_')

rename_map = {c: normalize(c) for c in df.columns if c != normalize(c)}
if rename_map:
    print('Renamed columns:')
    for old, new in rename_map.items():
        print(f'  {old!r}  ->  {new!r}')
    df = df.rename(columns=rename_map)
    if LABEL_COLUMN in rename_map:
        LABEL_COLUMN = rename_map[LABEL_COLUMN]

X = df.drop(columns=[LABEL_COLUMN])
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df[LABEL_COLUMN])
class_labels = [str(c) for c in label_encoder.classes_]
feature_columns = list(X.columns)

print(f'Features ({len(feature_columns)}): {feature_columns}')
print(f'Classes  ({len(class_labels)}):  {class_labels}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)
print(f'Train: {X_train.shape[0]} rows. Test: {X_test.shape[0]} rows.')

## 5. Train

One LightGBM classifier with the settings from above.

In [ ]:
import time

t0 = time.time()
model = lgb.LGBMClassifier(
    n_estimators=N_ESTIMATORS,
    learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH,
    random_state=RANDOM_SEED,
    verbose=-1,
)
model.fit(X_train, y_train)
print(f'Trained in {time.time() - t0:.1f} sec')

## 6. Evaluate

Accuracy, per-class precision / recall / F1, and a confusion matrix.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

y_pred = model.predict(X_test)
print(f'Test-set accuracy: {accuracy_score(y_test, y_pred):.3f}\n')
print(classification_report(y_test, y_pred, target_names=class_labels, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay(cm, display_labels=class_labels).plot(
    ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False
)
ax.set_title('Confusion Matrix (test set)')
plt.tight_layout()
plt.show()

## 7. Export to ONNX

Convert the LightGBM model to ONNX using `onnxmltools.convert_lightgbm`. Opset 18 first; falls back to 15 then 12 if needed.

In [ ]:
from onnxmltools.convert import convert_lightgbm
from onnxconverter_common.data_types import FloatTensorType
import onnx

initial_types = [('input', FloatTensorType([None, len(feature_columns)]))]

onnx_model = None
for opset in (18, 15, 12):
    try:
        onnx_model = convert_lightgbm(model, initial_types=initial_types, target_opset=opset)
        print(f'Exported at opset {opset}')
        break
    except Exception as ex:
        print(f'  opset {opset} failed ({type(ex).__name__}); trying lower')
if onnx_model is None:
    raise RuntimeError('All opset attempts failed.')

onnx.save_model(onnx_model, 'model.onnx')
print(f'Saved model.onnx ({len(onnx_model.SerializeToString()) / 1024:.1f} KB)')

## 8. Write sidecar files

IGNODE reads two sidecars alongside the model: `class_labels.json` (class names) and `feature_columns.json` (`{feature_columns, label_columns}`).

In [ ]:
import json

with open('class_labels.json', 'w') as f:
    json.dump(class_labels, f, indent=2)

feature_sidecar = {'feature_columns': feature_columns, 'label_columns': [LABEL_COLUMN]}
with open('feature_columns.json', 'w') as f:
    json.dump(feature_sidecar, f, indent=2)

print('class_labels.json:')
print(json.dumps(class_labels, indent=2))
print('\nfeature_columns.json:')
print(json.dumps(feature_sidecar, indent=2))

## 9. Download

In [ ]:
from google.colab import files
files.download('model.onnx')
files.download('class_labels.json')
files.download('feature_columns.json')

## 10. Upload to IGNODE

1. **Integrations → ML Factory** in your IGNODE portal
2. Switch to the **Custom Models** tab
3. Click **+ Upload ML Model**
4. Drop your `model.onnx` and fill in the metadata form:
    - **Task Type:** `Classification`
    - **Class Labels:** paste from `class_labels.json`
    - **Feature Columns:** paste from `feature_columns.json` → `feature_columns`
    - **Target Column:** paste from `feature_columns.json` → `label_columns` (single entry)
5. Click **Upload**, then **Open in Playground** to test.